# Evaluating zero-shot capabilites of pre-extracted features

In [ ]:
import sys
import torch
import torch.nn.functional as F
sys.path.append('../')

from src.evaluation.metrics import compute_multi_instance_recall
from src.datasets.dataset import EpicKitchensFeatureDataset
from torch.utils.data import DataLoader
from tqdm import tqdm


In [ ]:
features_path = "../data/features/clip/features_val.pt"

In [ ]:
val_dataset = EpicKitchensFeatureDataset(
    features_path = features_path,
    csv_path = "../data/annotations/processed/val.csv"
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

video_embeddings = []
text_embeddings = []
nouns = []
verbs = []

for batch in tqdm(val_loader):
    text, video, verb, noun = batch
    v_e = F.normalize(video, dim=1).detach().cpu()
    t_e = F.normalize(text, dim=1).detach().cpu()

    video_embeddings.append(v_e)
    text_embeddings.append(t_e)
    nouns.append(noun)
    verbs.append(verb)

In [ ]:
all_videos = torch.cat(video_embeddings)
all_texts = torch.cat(text_embeddings)

all_verbs = torch.cat(verbs)
all_nouns = torch.cat(nouns)

sim_matrix = torch.matmul(all_texts, all_videos.T)

recalls = compute_multi_instance_recall(sim_matrix, all_verbs, all_nouns, "val-zeroshot/")

for k, v in recalls.items():
    print(f"{k}: {v}")